# Hybrid Eigen-SVD + EfficientNet-B0 + Multi-Scale SE/CA/SA Attention — CIFAR-100

Pipeline: image -> **EigenTransform** (live, differentiable, per-channel top-k SVD reconstruction, replaces raw pixels) -> **EfficientNet-B0** (trained from scratch, since the eigen-transformed input has nothing to do with ImageNet pixel statistics) -> three feature maps tapped at three EfficientNet-B0 stages -> **A1 / A2 / A3**: a different lightweight attention mechanism per branch -- A1 uses Squeeze-and-Excitation, A2 uses Channel Attention (CBAM-style, avg+max pooled), A3 uses Spatial Attention (CBAM-style, per-pixel gating) -> **Global Attention Pooling** (learned, softmax-weighted, not a plain average) per branch -> concat -> classifier.

**Note on resolution:** CIFAR-100 images are natively 32x32. This notebook resizes them up to 224x224 to match EfficientNet-B0's stage taps used elsewhere in the design. Unlike the earlier Swin-attention version, SE/CA/SA place no divisibility constraint on resolution -- they operate on the full feature map at whatever size it is -- but the resize is still a real compute cost (each image upsampled 7x per side, and the live per-image SVD scales with resolution too). One thing that softens this: since the *source* images are only 32x32, the resized 224x224 version rarely has genuine information above roughly rank 32 anyway, so `eigen_rank=32` isn't as aggressive a truncation as it would be on a real 224x224 photo.

In [1]:
!pip install timm -q

In [2]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import timm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## Data — CIFAR-100

Resized to 224x224 for the reasons above. No CIFAR mean/std normalization is applied here — the model's `post_eigen_norm` (a `BatchNorm2d` right after the eigen-transform) learns its own scale for whatever the eigen-transform outputs, so pre-normalizing raw pixels first would just be undone anyway.

In [3]:
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomCrop(IMG_SIZE, padding=16, padding_mode="reflect"),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

In [4]:
DATA_ROOT = "./data"
os.makedirs(DATA_ROOT, exist_ok=True)
train_full_aug = datasets.CIFAR100(root=DATA_ROOT, train=True, download=True, transform=train_transform)
train_full_eval = datasets.CIFAR100(root=DATA_ROOT, train=True, download=True, transform=eval_transform)
test_ds = datasets.CIFAR100(root=DATA_ROOT, train=False, download=True, transform=eval_transform)

Files already downloaded and verified


D:\CODE\Thesis\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Files already downloaded and verified
Files already downloaded and verified


## Train / validation split

A held-out slice of the training set, evaluated without augmentation (`train_full_eval`'s transform) so validation accuracy isn't inflated or noised by augmentation.

In [5]:
VAL_FRACTION = 0.1
num_train_total = len(train_full_aug)
indices = list(range(num_train_total))
random.Random(SEED).shuffle(indices)
num_val = int(num_train_total * VAL_FRACTION)
val_indices = indices[:num_val]
train_indices = indices[num_val:]

train_ds = Subset(train_full_aug, train_indices)
val_ds = Subset(train_full_eval, val_indices)

print(f"train: {len(train_ds)}  val: {len(val_ds)}  test: {len(test_ds)}")

train: 45000  val: 5000  test: 10000


In [7]:
BATCH_SIZE = 32
NUM_WORKERS = 8  # tune to your CPU's core count

# persistent_workers=True avoids respawning the worker processes at the start
# of every single epoch -- on Windows (spawn-based multiprocessing) that
# respawn cost is real and adds dead time before each epoch even starts.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                           pin_memory=True, drop_last=True, persistent_workers=(NUM_WORKERS > 0))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                         pin_memory=True, persistent_workers=(NUM_WORKERS > 0))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                          pin_memory=True, persistent_workers=(NUM_WORKERS > 0))

## Model

Same eigen-transform and EfficientNet-B0 backbone as before. A1/A2/A3 now each use a different lightweight attention mechanism instead of Swin blocks: A1 = Squeeze-and-Excitation, A2 = Channel Attention (CBAM-style), A3 = Spatial Attention (CBAM-style). None of these need a resolution-divisibility constraint -- they operate on whatever (B, C, H, W) feature map they're given.

In [8]:
class EigenTransform(nn.Module):
    """Live, differentiable per-channel low-rank SVD reconstruction. Replaces
    the raw pixel input; same (B, C, H, W) shape in/out.

    Uses torch.svd_lowrank (randomized, targets rank k directly) rather than
    a full economy SVD followed by truncation. Both target the same object
    -- the best rank-k approximation -- but full-SVD-then-truncate wastefully
    computes H-k singular components it immediately discards. On structured,
    image-like data the two give reconstruction error within ~0.1% of each
    other; on GPU, svd_lowrank is also far cheaper, since it is built from
    matmuls and QR (which parallelize well) rather than the sequential
    bidiagonalization full SVD relies on.

    Forced to run in fp32 regardless of any surrounding autocast (mixed
    precision) context, since the randomized SVD is not numerically safe in
    fp16/bf16.
    """
    def __init__(self, rank: int = 32, niter: int = 4):
        super().__init__()
        self.rank = rank
        self.niter = niter

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        k = min(self.rank, H, W)
        x_flat = x.reshape(B * C, H, W)
        with torch.autocast(device_type=x.device.type, enabled=False):
            x_flat_fp32 = x_flat.float()
            U, S, V = torch.svd_lowrank(x_flat_fp32, q=k, niter=self.niter)
            recon = torch.einsum("bhk,bk,bkw->bhw", U, S, V.transpose(-2, -1))
        return recon.reshape(B, C, H, W).to(x.dtype)


class SEBlock(nn.Module):
    """Squeeze-and-Excitation (Hu et al. 2018): global-average-pool
    'squeeze' to one descriptor per channel, a small bottleneck MLP
    'excites' it back out to a per-channel gate in [0, 1], which rescales
    the original feature map channel-wise."""
    def __init__(self, dim: int, reduction: int = 16):
        super().__init__()
        hidden = max(dim // reduction, 4)
        self.fc = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, dim),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        squeeze = x.mean(dim=(2, 3))
        gate = self.fc(squeeze).view(B, C, 1, 1)
        return x * gate


class ChannelAttention(nn.Module):
    """CBAM-style channel attention: unlike plain SE, uses BOTH average-pool
    AND max-pool descriptors (each passed through the same shared MLP, then
    summed) before the sigmoid gate -- the max-pool branch captures
    'is this channel strongly present anywhere', which average pooling
    alone can dilute."""
    def __init__(self, dim: int, reduction: int = 16):
        super().__init__()
        hidden = max(dim // reduction, 4)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, dim),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        avg_pool = x.mean(dim=(2, 3))
        max_pool = x.amax(dim=(2, 3))
        gate = self.sigmoid(self.mlp(avg_pool) + self.mlp(max_pool)).view(B, C, 1, 1)
        return x * gate


class SpatialAttention(nn.Module):
    """CBAM-style spatial attention: pools ACROSS channels (not across
    space, unlike SE/CA) to get one avg map and one max map, concatenates
    them, and runs a conv to produce a single per-pixel gate -- this asks
    'which locations matter', a different question than channel attention's
    'which channels matter'."""
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        avg_out = x.mean(dim=1, keepdim=True)
        max_out = x.amax(dim=1, keepdim=True)
        concat = torch.cat([avg_out, max_out], dim=1)
        gate = self.sigmoid(self.conv(concat))
        return x * gate


class AttentionPool(nn.Module):
    """Global Attention Pooling: learned, softmax-weighted sum over tokens."""
    def __init__(self, dim: int):
        super().__init__()
        self.score = nn.Linear(dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        weights = torch.softmax(self.score(x), dim=1)
        return (weights * x).sum(dim=1)


class HybridEigenAttnNet(nn.Module):
    def __init__(self, num_classes: int = 100, eigen_rank: int = 32, embed_dim: int = 192,
                 reduction: int = 16, sa_kernel_size: int = 7, img_size: int = 224):
        super().__init__()
        self.eigen = EigenTransform(rank=eigen_rank)
        self.post_eigen_norm = nn.BatchNorm2d(3)

        self.backbone = timm.create_model(
            "efficientnet_b0", pretrained=False, features_only=True, out_indices=(2, 3, 4)
        )
        stage_channels = self.backbone.feature_info.channels()

        self.proj = nn.ModuleList([nn.Conv2d(c, embed_dim, kernel_size=1) for c in stage_channels])

        # one distinct attention mechanism per branch, in place of Swin blocks --
        # none of these need a resolution-divisibility constraint
        self.attn_blocks = nn.ModuleList([
            SEBlock(embed_dim, reduction=reduction),           # A1: Squeeze-and-Excitation
            ChannelAttention(embed_dim, reduction=reduction),   # A2: Channel Attention
            SpatialAttention(kernel_size=sa_kernel_size),        # A3: Spatial Attention
        ])
        self.branch_names = ["A1 (SE)", "A2 (CA)", "A3 (SA)"]

        self.pools = nn.ModuleList([AttentionPool(embed_dim) for _ in stage_channels])
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim * len(stage_channels)),
            nn.Linear(embed_dim * len(stage_channels), num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.eigen(x)
        x = self.post_eigen_norm(x)
        feats = self.backbone(x)
        branch_outs = []
        for feat, proj, attn, pool in zip(feats, self.proj, self.attn_blocks, self.pools):
            f = proj(feat)
            f = attn(f)
            tokens = f.flatten(2).transpose(1, 2)
            branch_outs.append(pool(tokens))
        fused = torch.cat(branch_outs, dim=1)
        return self.classifier(fused)

In [9]:
NUM_CLASSES = 100  # CIFAR-100

model = HybridEigenAttnNet(
    num_classes=NUM_CLASSES,
    eigen_rank=32,
    embed_dim=192,
    reduction=16,
    sa_kernel_size=7,
    img_size=IMG_SIZE,
).to(device)

for name, attn in zip(model.branch_names, model.attn_blocks):
    print(f"{name}: {type(attn).__name__}, {sum(p.numel() for p in attn.parameters()):,} params")

n_params = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {n_params:,}")

# sanity check on one real batch before committing to a full training run
xb, yb = next(iter(train_loader))
xb, yb = xb.to(device), yb.to(device)
out = model(xb)
print("output shape:", out.shape)

A1 (SE): SEBlock, 4,812 params
A2 (CA): ChannelAttention, 4,812 params
A3 (SA): SpatialAttention, 98 params
Total trainable parameters: 3,755,747
output shape: torch.Size([32, 100])


## Optimizer, scheduler, and a quick benchmark before committing to a full run

Set up here (rather than right before the training loop) so the benchmark cell below can use the real `optimizer`/`scaler`/`criterion` instead of standing up throwaway copies.

In [10]:
EPOCHS = 30
LR = 3e-4
WEIGHT_DECAY = 0.05
USE_AMP = (device.type == "cuda")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)

# Mixed precision speeds up the conv/attention/matmul-heavy parts of the model
# on Tensor Cores (roughly halves their cost) and is standard, accuracy-neutral
# practice. EigenTransform ignores this and always runs its SVD in fp32
# internally regardless of this autocast context, so numerical stability of
# the eigen-transform itself is unaffected either way.

Compares directly against the earlier measurement of ~2265 ms/step (~53 min/epoch). The two changes behind this cell: `EigenTransform` now uses `torch.svd_lowrank` instead of a full SVD it then truncated (it was computing all 224 singular values every step and discarding 192 of them -- the actual cause of the slowdown), and the training step now runs under mixed precision. Neither should meaningfully change final accuracy: `svd_lowrank` targets the same rank-32 approximation, verified above to land within ~0.1% reconstruction error of the full-SVD version on structured data, and mixed precision is standard, accuracy-neutral practice.

In [11]:
import time

model.train()
data_iter = iter(train_loader)

for _ in range(3):  # warmup: skips one-time CUDA kernel compilation / worker startup cost
    xb, yb = next(data_iter)
    xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
    optimizer.zero_grad()
    with torch.amp.autocast(device.type, enabled=USE_AMP):
        loss = criterion(model(xb), yb)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

if device.type == "cuda":
    torch.cuda.synchronize()
n_steps = 20
t0 = time.time()
for _ in range(n_steps):
    xb, yb = next(data_iter)
    xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
    optimizer.zero_grad()
    with torch.amp.autocast(device.type, enabled=USE_AMP):
        loss = criterion(model(xb), yb)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
if device.type == "cuda":
    torch.cuda.synchronize()

per_step = (time.time() - t0) / n_steps
print(f"{per_step*1000:.0f} ms/step -> ~{per_step*len(train_loader)/60:.1f} min for one training epoch")

262 ms/step -> ~6.1 min for one training epoch


If that estimate is still above 5 minutes, the next lever (raising `BATCH_SIZE`, since AMP roughly halves memory use and there's likely headroom now) is a quick thing to try before anything more invasive -- larger batches reduce the relative overhead per step. Let me know the number and I'll help from there rather than guessing further changes blind.

## Training loop

In [12]:
def run_epoch(loader, train: bool):
    model.train(mode=train)
    total_loss, total_correct, total_count = 0.0, 0, 0
    torch.set_grad_enabled(train)
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        if train:
            optimizer.zero_grad()
        try:
            with torch.amp.autocast(device.type, enabled=USE_AMP):
                out = model(xb)
                loss = criterion(out, yb)
            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print(f"OOM at batch size {xb.size(0)} -- lower BATCH_SIZE and rerun.")
            raise
        total_loss += loss.item() * xb.size(0)
        total_correct += (out.argmax(dim=1) == yb).sum().item()
        total_count += xb.size(0)
    torch.set_grad_enabled(True)
    return total_loss / total_count, total_correct / total_count


best_val_acc = 0.0
epochs_without_improvement = 0
PATIENCE = 7  # epochs without val_acc improvement before stopping early

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    scheduler.step()
    print(f"epoch {epoch:02d}/{EPOCHS}  train_loss {train_loss:.4f}  train_acc {train_acc:.4f}  "
          f"val_loss {val_loss:.4f}  val_acc {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_without_improvement = 0
        torch.save(model.state_dict(), "best_hybrid_eigen_model.pt")
    else:
        epochs_without_improvement += 1
        print(f"  no improvement for {epochs_without_improvement}/{PATIENCE} epoch(s)")
        if epochs_without_improvement >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (best val_acc={best_val_acc:.4f})")
            break

print("best val acc:", best_val_acc)

epoch 01/30  train_loss 3.7658  train_acc 0.1573  val_loss 3.2359  val_acc 0.2704
epoch 02/30  train_loss 3.0386  train_acc 0.3234  val_loss 2.7733  val_acc 0.3848
epoch 03/30  train_loss 2.6445  train_acc 0.4300  val_loss 2.4988  val_acc 0.4718
epoch 04/30  train_loss 2.3902  train_acc 0.5036  val_loss 2.3268  val_acc 0.5160
epoch 05/30  train_loss 2.2004  train_acc 0.5596  val_loss 2.2337  val_acc 0.5484
epoch 06/30  train_loss 2.0502  train_acc 0.6056  val_loss 2.1508  val_acc 0.5748
epoch 07/30  train_loss 1.9250  train_acc 0.6464  val_loss 2.0740  val_acc 0.6000
epoch 08/30  train_loss 1.8149  train_acc 0.6801  val_loss 2.0404  val_acc 0.6052
epoch 09/30  train_loss 1.7182  train_acc 0.7124  val_loss 2.0001  val_acc 0.6164
epoch 10/30  train_loss 1.6263  train_acc 0.7448  val_loss 1.9931  val_acc 0.6296
epoch 11/30  train_loss 1.5435  train_acc 0.7737  val_loss 1.9206  val_acc 0.6478
epoch 12/30  train_loss 1.4695  train_acc 0.8003  val_loss 1.9314  val_acc 0.6552
epoch 13/30  tra

## Test evaluation

In [13]:
model.load_state_dict(torch.load("best_hybrid_eigen_model.pt", map_location=device))
test_loss, test_acc = run_epoch(test_loader, train=False)
print(f"test_loss {test_loss:.4f}  test_acc {test_acc:.4f}")

C:\Users\adnan\AppData\Local\Temp\ipykernel_2804\3971303063.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_hybrid_eigen_model.pt"

test_loss 1.8396  test_acc 0.6969


In [14]:
import torch

device = next(model.parameters()).device
x = torch.randn(4, 3, 224, 224, device=device)

with torch.no_grad():
    out = model.eigen(x)

print(f"input shape:  {tuple(x.shape)}")
print(f"output shape: {tuple(out.shape)}")
print(f"shapes match: {out.shape == x.shape}")
print(f"dtype match:  {out.dtype == x.dtype}  (in={x.dtype}, out={out.dtype})")
print(f"has NaN:      {torch.isnan(out).any().item()}")
print(f"has Inf:      {torch.isinf(out).any().item()}")

assert out.shape == x.shape, f"shape mismatch! expected {x.shape}, got {out.shape}"
print("\nEigenTransform shape check: PASSED")

# bonus: confirm it's actually doing a low-rank reconstruction, not just
# preserving shape trivially
svals = torch.linalg.svdvals(out[0, 0])  # one image, one channel
effective_rank = (svals > svals.max() * 1e-4).sum().item()
print(f"effective rank of that channel: {effective_rank}  (eigen_rank={model.eigen.rank})")

input shape:  (4, 3, 224, 224)
output shape: (4, 3, 224, 224)
shapes match: True
dtype match:  True  (in=torch.float32, out=torch.float32)
has NaN:      False
has Inf:      False

EigenTransform shape check: PASSED
effective rank of that channel: 32  (eigen_rank=32)


In [15]:
import torch

device = next(model.parameters()).device
torch.set_printoptions(precision=3, sci_mode=False, linewidth=120)

xb, yb = next(iter(val_loader))
xb = xb[:1].to(device)  # one image, keeps the printout readable

with torch.no_grad():
    x_eigen = model.eigen(xb)                    # after EigenTransform
    x_cnn_input = model.post_eigen_norm(x_eigen)  # after BatchNorm -- what EfficientNet-B0 actually sees

img, ch, row, col = 0, 0, 100, 100  # first image, red channel, a 6x6 patch away from the edges
sl = slice(row, row + 6)

print(f"tensor shape at each stage: {tuple(xb.shape)}\n")

print("1) RAW pixels (before any transform):")
print(xb[img, ch, sl, sl])

print("\n2) AFTER eigen transform (rank-{} SVD reconstruction):".format(model.eigen.rank))
print(x_eigen[img, ch, sl, sl])

print("\n3) AFTER post_eigen_norm (actual CNN input):")
print(x_cnn_input[img, ch, sl, sl])

print("\ndifference, raw vs eigen (same patch):")
print(xb[img, ch, sl, sl] - x_eigen[img, ch, sl, sl])

for name, t in [("raw", xb[img, ch]), ("eigen", x_eigen[img, ch]), ("cnn_input", x_cnn_input[img, ch])]:
    print(f"\n{name:10s} -> min {t.min():.4f}  max {t.max():.4f}  mean {t.mean():.4f}  std {t.std():.4f}")

tensor shape at each stage: (1, 3, 224, 224)

1) RAW pixels (before any transform):
tensor([[1.000, 1.000, 0.996, 0.992, 0.988, 0.988],
        [1.000, 1.000, 0.996, 0.992, 0.988, 0.988],
        [1.000, 1.000, 0.992, 0.988, 0.984, 0.980],
        [0.996, 0.996, 0.992, 0.984, 0.980, 0.976],
        [0.996, 0.996, 0.988, 0.980, 0.976, 0.969],
        [0.996, 0.996, 0.988, 0.980, 0.969, 0.961]], device='cuda:0')

2) AFTER eigen transform (rank-32 SVD reconstruction):
tensor([[0.999, 0.999, 0.996, 0.993, 0.990, 0.990],
        [1.000, 1.000, 0.996, 0.992, 0.989, 0.988],
        [1.000, 0.999, 0.994, 0.989, 0.985, 0.981],
        [0.996, 0.996, 0.991, 0.984, 0.980, 0.975],
        [0.996, 0.996, 0.989, 0.981, 0.975, 0.968],
        [0.996, 0.996, 0.987, 0.980, 0.971, 0.962]], device='cuda:0')

3) AFTER post_eigen_norm (actual CNN input):
tensor([[1.241, 1.240, 1.231, 1.222, 1.215, 1.214],
        [1.242, 1.242, 1.231, 1.221, 1.213, 1.209],
        [1.241, 1.240, 1.226, 1.212, 1.200, 1.190]